# tofu_v9 — referee-response campaign driver

Run the cells **top to bottom**. Cells 1–6 are setup (run at the start of every session — VMs are wiped between sessions). Cell 7 is session A (~2.5 h), Cell 8 session B (~6–7 h), Cell 9 the wrap-up.

Resumable: if the VM dies, reconnect, run Cells 1–6 again, then rerun the driver cell you were in — finished stages skip themselves.


### Cell 1 — mount Drive
Run first, every session. Everything below assumes Drive is at /content/drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 2 — packages
Fresh VMs need the libraries. If your notebook already has its own install cell that has worked before, run that instead — this is the equivalent.


In [ ]:
%pip install -q -U transformers peft accelerate datasets bitsandbytes sentencepiece rouge-score


### Cell 3 — copy the scripts to the VM
Pulls the campaign scripts from your NMI_tofu folder (tries both places it might live). All eight should list at the end.


In [ ]:
import os
SRC = "/content/drive/MyDrive/Persistence of Memory/NMI_tofu"
if not os.path.isdir(SRC):
    SRC = "/content/drive/MyDrive/tofu_v3_backup/NMI_tofu"
for f in ["tofu_all.py", "tofu_v3.py", "tofu_v4.py", "tofu_v5.py",
          "tofu_v6.py", "tofu_v7.py", "tofu_v8.py", "tofu_v9.py"]:
    !cp "{SRC}/{f}" /content/
!ls /content/*.py

### Cell 4 — restore the adapters v9 needs
Targeted copy: only the adapters v9 uses (a few hundred MB), not the multi-GB full backup. cp prints each file so you can see progress. Done when the ls shows full, retain90, unlearn_neggrad.


In [ ]:
import os
need = ([f"s{i}/retain90" for i in range(10)] +
        [f"s{i}/{d}" for i in "012" for d in ["full", "unlearn_neggrad"]])
for d in need:
    os.makedirs(f"/content/adapters/{d.split('/')[0]}", exist_ok=True)
    !cp -rv "/content/drive/MyDrive/tofu_v3_backup/adapters/{d}" "/content/adapters/{d}"
!ls /content/adapters/s0/

### Cell 5 — load the library (tofu_all through v8)
Defines every function v9 reuses. With no stage argument each script just prints its usage text and defines its functions — nothing trains here. Takes a minute or two (model libraries import).


In [ ]:
import sys, os
os.chdir("/content")
for f in ["tofu_all.py", "tofu_v3.py", "tofu_v4.py", "tofu_v5.py",
          "tofu_v6.py", "tofu_v7.py", "tofu_v8.py"]:
    print("loading", f, flush=True)
    sys.argv = [f]
    exec(open(f).read())
print("library loaded")

### Cell 6 — define backup()
Per-stage backup to Drive, same idea as the v5/v6 driver. Copies results_v9 and any NEW adapter dirs v9 creates. A VM recycle then costs nothing — rerunning skips finished stages.


In [ ]:
import subprocess
def backup():
    subprocess.run(["bash", "-c",
      "mkdir -p '/content/drive/MyDrive/tofu_v3_backup/results_v9' && "
      "cp -r results_v9/. '/content/drive/MyDrive/tofu_v3_backup/results_v9/' 2>/dev/null; "
      "for d in adapters/s*/benignrec9 adapters/s*/benign_of_full9 adapters/s*/oraclex "
      "adapters/s*/oraclehi adapters/s*/oraclerep adapters/s*/rmufull_a adapters/s*/rmufull_b "
      "adapters/s*/attr_distill; do "
      "  if [ -d \"$d\" ]; then mkdir -p \"/content/drive/MyDrive/tofu_v3_backup/$d\"; "
      "  cp -rn \"$d/.\" \"/content/drive/MyDrive/tofu_v3_backup/$d/\"; fi; done; true"])
    print("backed up results_v9 + new adapters to Drive")
backup

### Cell 7 — driver, session A (~2.5 h)
The free-standing referee answers: held-out relearning folds (referee 3), probe robustness (referee 5), logit-lens layer probe and update-cosine nulls (referee 2). Safe to stop and rerun — every stage skips existing outputs.


In [ ]:
import sys
STAGES_A = ([("relearn95", s) for s in "012"] + [("probe9", "0")] +
            [("lens", s) for s in "012"] + [("nulls", s) for s in "012"])
for stage, seed in STAGES_A:
    print(f"\n===== v9 {stage} {seed} =====", flush=True)
    sys.argv = ["tofu_v9.py", stage, seed]
    exec(open("tofu_v9.py").read())
    backup()
print("SESSION A COMPLETE")

### Cell 8 — driver, session B (~6-7 h)
The heavy arms: oracle at 10x budget plus lr control (referee 6), representation oracle all seeds, canonical full-parameter RMU (referee 7), distillation seeds 1-2 (referee 8), MUSE-QA seeds 1-2 (referee 4). Can run in the same session as A if the VM holds, or a fresh session after rerunning Cells 1-6.


In [ ]:
!cp -r /content/drive/MyDrive/tofu_v3_backup/adapters/museqa /content/adapters/ 2>/dev/null
import sys
STAGES_B = ([("oraclex", s) for s in "012"] + [("oraclehi", "0")] +
            [("oraclerep", s) for s in "12"] + [("rmufull", s) for s in "012"] +
            [("distill", s) for s in "12"] + [("museqa", s) for s in "12"])
for stage, seed in STAGES_B:
    print(f"\n===== v9 {stage} {seed} =====", flush=True)
    sys.argv = ["tofu_v9.py", stage, seed]
    exec(open("tofu_v9.py").read())
    backup()
print("SESSION B COMPLETE")

### Cell 9 — wrap up: summary + zip to Drive
Prints the one-line summary of every result, zips results_v9, and puts the zip at the top of MyDrive. Download it into the project folder on your Mac and tell Claude it's there.


In [ ]:
import sys
sys.argv = ["tofu_v9.py", "figures9"]
exec(open("tofu_v9.py").read())
!cd /content && zip -qr results_v9.zip results_v9
!cp /content/results_v9.zip "/content/drive/MyDrive/"
print("results_v9.zip is in MyDrive — download it into the Persistence of Memory folder")